# Measuring Teacher Attrition with Labour Force Survey Data — full replication

Replicates every number and figure of the Data Cluster deck (both the Beamer PDF and the PowerPoint), from the raw public files to the final slides.

## What you need
- Python 3.11+ with: `pandas numpy scipy statsmodels matplotlib openpyxl pyarrow adjustText python-pptx Pillow`
- `pdflatex` (TeX Live) and `pdftotext`/`pdftoppm` (poppler) for the Beamer deck and the equation image
- ~12 GB of free disk and a few hours (the ASEC and monthly CPS downloads dominate)
- Internet access to census.gov and nber.org

## What gets downloaded (all public, no registration)
| Data | Years | Source | Used for |
|---|---|---|---|
| ASEC (March CPS) fixed-width + technical docs | 1998–2010 | `www2.census.gov/programs-surveys/cps/datasets/{year}/march/` | master database (byte positions read from the official PDFs) |
| ASEC machine-readable | 2011–2025 | same | master database |
| ASEC public-use CSV (`asecpubYYcsv.zip`) | 2019–2025 | same | extended covariates (pension, health, disability, parents, income) |
| CPS basic monthly | 2005–2025 | `data.nber.org/cps-basic3` (to 2023), `census.gov .../basic/` (2024+) | rotation checks, person panel, linkage rates, six definitions |
| Geographic mobility table A-1 (`hst_mig_a_1.xlsx`) | 1948–2023 | `www2.census.gov/programs-surveys/demo/tables/geographic-mobility/time-series/historic/` | internal migration chart |
| CPI-U (CPIAUCSL) | 1947– | ships precomputed in `outputs/n_cpi.csv` (FRED) | real pay series |
| EB Garamond (OTF) | — | `mirrors.ctan.org/fonts/ebgaramond/opentype/` | chart typography |

External benchmarks quoted on slides (NCES TFS/NTPS, OECD *Education at a Glance* 2021/2025/2026, Harris & Adams 2007, Aldeman & Yi 2025) are cited, not computed.

## 0. Get the code

In [ ]:
# clone the repository AT THE RIGHT BRANCH (the repo default branch is an older one)
!git clone -b claude/teacher-leave-replication-43x62u https://github.com/jedr1-ship-it/Alternative-Pathways----Quant.git
%cd Alternative-Pathways----Quant
!pip -q install pandas numpy scipy statsmodels matplotlib openpyxl pyarrow adjustText python-pptx Pillow

## 1. Master database, 1998–2025 (~5.0M person-years)
Downloads the ASEC files and official documentation itself, extracts the 1998–2010 fixed-width supplements at the documented byte positions, pools them with the 2011–2025 machine-readable files, and writes `data/processed/asec_master.parquet` (5,009,129 records).

In [ ]:
!python replication/build_database.py

## 2. Teacher files
`teachers.parquet` (every teacher-year: longest job last year was K-12 teaching) and `teachers_main.parquet` (main sample: college graduates 18+, valid weight and outcome).

In [ ]:
!python replication/build_teacher_sample.py

## 3. Extended ASEC covariates, 2019–2025
Downloads the public-use CSVs and builds `data/processed/asec_ext.parquet` (pension plan, class of worker, health, disability, parent pointers, household income), then computes the unified flow (`outputs/flow_unified.csv`), the LPM/logit/probit results and the boundedness check.

In [ ]:
import os, subprocess
os.makedirs("data/raw/aseccsv", exist_ok=True)
for y in range(19, 26):
    url = f"https://www2.census.gov/programs-surveys/cps/datasets/20{y}/march/asecpub{y}csv.zip"
    dest = f"data/raw/aseccsv/asecpub{y}csv.zip"
    if not os.path.exists(dest):
        print("downloading", url)
        subprocess.run(["curl", "-sL", "--retry", "3", url, "-o", dest], check=True)
!python replication/build_asec_ext.py
!python replication/analyze_ext.py

## 4. CPS basic monthly, 2005–2025 (optional but needed for the panel results)
251 monthly files (~6 GB). Feeds the rotation-group checks, the person panel (`build_panel_person_r3.py`), the linkage rates and the six-definitions ladder. Identifiers: household id + dwelling id (HRHHID2, from May 2004) + person line number, validated on sex, race, age +0–2 (Madrian & Lefgren 1999).

In [ ]:
!python replication/build_cps_monthly.py
!python replication/build_panel_person_r3.py

## 5. Auxiliary downloads

In [ ]:
import os, subprocess
# Census historical migration table A-1
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/census_mobility_a1.xlsx"):
    subprocess.run(["curl", "-sL",
        "https://www2.census.gov/programs-surveys/demo/tables/geographic-mobility/time-series/historic/hst_mig_a_1.xlsx",
        "-o", "data/raw/census_mobility_a1.xlsx"], check=True)
# EB Garamond for the charts (install system-wide or point matplotlib at the files)
os.makedirs("fonts_eb", exist_ok=True)
for f in ["EBGaramond-Regular", "EBGaramond-Bold", "EBGaramond-Italic", "EBGaramond-SemiBold"]:
    dest = f"fonts_eb/{f}.otf"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sL",
            f"https://mirrors.ctan.org/fonts/ebgaramond/opentype/{f}.otf", "-o", dest], check=True)
import matplotlib.font_manager as fm
for f in os.listdir("fonts_eb"):
    fm.fontManager.addfont(os.path.join("fonts_eb", f))
print("fonts registered")

## 6. Rebuild every figure
`make_deck_figures.py` runs the original 21 scripts; the loop below adds the Data-Cluster-specific builders (five-year flows, study cascade, appendix charts, AER inputs, comparability with counselors).

In [ ]:
!python replication/make_deck_figures.py
for s in ["build_newdeck_figs", "build_appendix_figs", "build_flow5yr_figs",
          "build_study_fig", "build_flow_lpm_figs", "fig10_all_occupations",
          "fig17_pay_index", "fig18_pay_deciles", "fig22_pension_states",
          "fig_sighting_gradient", "fig04_05_professions",
          "fig13_age_route_profiles", "fig20_earnings_by_destination",
          "fig02_leaving_series", "fig03_sector"]:
    print("==>", s)
    !python replication/deck_figures/{s}.py

## 7. Rebuild the two decks
The Beamer PDF is the master; the PowerPoint is built natively (Garamond, editable text and tables, working appendix buttons) from the same figure PDFs rendered at 300 dpi.

In [ ]:
import glob, os, subprocess
# copy figure PDFs used by the deck and render 300-dpi PNGs for the PPTX
os.makedirs("report/data_cluster/_pngx", exist_ok=True)
FIGS = ["defs6","g2_evolution","leave_professions_deck","flow_employed_5yr",
        "occ_all_slide","flow_olf55_5yr","study_olf55","ame_ext","msgA3",
        "pay3_deciles_deck","pension_scatter_deck","cycle1","tfs_waves",
        "migration_rate","linkage_series","sighting_gradient","pubpriv_series",
        "profs_ext","codes_narrow","lpm_check","u_routes_deck",
        "pay7_destinations_deck"]
for f in FIGS:
    subprocess.run(["cp", f"report/figures_deck/{f}.pdf", "report/data_cluster/figures/"], check=True)
    subprocess.run(["pdftoppm", "-png", "-r", "300", "-singlefile",
                    f"report/data_cluster/figures/{f}.pdf",
                    f"report/data_cluster/_pngx/{f}"], check=True)
# Beamer (run twice for the internal links)
subprocess.run(["pdflatex", "-interaction=nonstopmode", "main.tex"],
               cwd="report/data_cluster", check=False)
subprocess.run(["pdflatex", "-interaction=nonstopmode", "main.tex"],
               cwd="report/data_cluster", check=True)
# native editable PowerPoint
subprocess.run(["python", "replication/deck_figures/build_cluster_pptx.py"], check=True)
print("decks written: report/data_cluster/main.pdf and Proyect_US_cluster.pptx")

## 8. Verify the headline numbers
Expected values (calendar years, weighted):
- Leaving rate 2024: **7.7%**; 1997–2024 average **8.6** (`outputs/p_series.csv`)
- Of every 100 leavers 2020–2024: another job **28.8**, retired/OLF 55+ **36.4**, OLF under 55 **23.7**, unemployed **11.1** (`outputs/flow_unified.csv`)
- Probit AMEs (2018–2024, n=18,571): full-year contract **−8.5**, part-time **+4.3**, pension **−2.4**, new baby **+4.6**, disability **+4.2**, bottom-quartile income **+2.5** (`outputs/models_table.csv`)
- LPM fitted values below zero: **22%** → probit is the main specification

In [ ]:
import pandas as pd
print(pd.read_csv("outputs/p_series.csv")[["cal_year","leaver_ba"]].tail(3).to_string(index=False))
print()
print(pd.read_csv("outputs/flow_unified.csv", index_col=0).iloc[:4,0].round(1).to_string())
print()
m = pd.read_csv("outputs/models_table.csv").set_index("var")
print(m.loc[["fullyear","parttime","pension","baby","disab","lowinc"],
            ["probit","probit_se"]].round(2).to_string())
print()
print(pd.read_csv("outputs/lpm_fitted_check.csv", index_col=0).round(3).to_string())